# MCPGuard — exploratory analysis & results walkthrough
This notebook reads the artefacts produced by `python -m src.pipeline.run_all`; it does not retrain anything. Run it from the repository root (or keep the `sys.path` cell).

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'src').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.training.data import load_with_split
df = load_with_split()
df.shape, df['error_occurred'].mean()

## Dataset composition and provenance

In [ ]:
pd.crosstab(df['source'], df['error_severity'], margins=True)

In [ ]:
df.groupby('source')[['failure_rate','parameter_count','description_length']].describe().T.round(2)

## Engineered risk scores vs. outcome

In [ ]:
st = pd.read_csv(ROOT / 'data/processed/structured_features.csv')
risk = ['ambiguity_score','missing_constraint_score','parameter_complexity_score','schema_complexity_score','description_quality_score','instruction_clarity_score']
st.groupby('error_occurred')[risk].mean().T.round(3)

## Benchmark (test split) — read from artifacts/metrics/benchmark.json

In [ ]:
bench = json.loads((ROOT / 'artifacts/metrics/benchmark.json').read_text())
pd.DataFrame([{'id': r['id'], 'model': r['name'], **{k: round(r['test'][k], 3) for k in ('accuracy','precision','recall','f1','roc_auc','pr_auc')}} for r in bench['results']]).sort_values('pr_auc', ascending=False)

## Ablation

In [ ]:
ab = json.loads((ROOT / 'artifacts/metrics/ablation.json').read_text())
pd.DataFrame([{'set': r['set'], 'features': r['n_features'], 'cv_pr_auc': round(r['cv']['pr_auc']['mean'], 3), 'sd': round(r['cv']['pr_auc']['std'], 3), 'test_pr_auc': round(r['test']['pr_auc'], 3)} for r in ab['rows']])

## Real-time inference with explanation

In [ ]:
from src.inference.predictor import predict_one
r = predict_one({'tool_name': 'process', 'description': 'Handles data operations as needed.', 'input_schema': {'type': 'object', 'properties': {'data': {}, 'mode': {}}}})
{k: r[k] for k in ('probability','risk_level','risk_factors')}, [x['title'] for x in r['recommendations']]

## Figures
All figures are written to `artifacts/figures/` by `python -m src.visualization.figures`.

In [ ]:
from IPython.display import Image
Image(filename=str(ROOT / 'artifacts/figures/03_pr_curves.png'))